In [1]:
import astropy.units as u
import pandas as pd
import logging
import re

In [2]:
import scopesim as sim

In [ ]:
# NB: should pull from scopesim and irdb githubs together
sim.rc.__config__["!SIM.file.local_packages_path"] = "/home/jessicap/Desktop/Packages/irdb"

In [4]:
# From what I can tell (which is very little at the moment) the way to 
# record the calculated DIT if it is under MINDIT, is to pull it from the 
# stdout warning that scopesim prints

# This is just setting up the system to record the output of the warning
class DebugCapture(logging.Handler):
    def __init__(self):
        super().__init__()
        self.raw = []

    def emit(self, record):
        self.raw.append(record.getMessage())

debug_capture = DebugCapture()

logger = logging.getLogger('astar.scopesim.effects.electronic')
print("effective level:", logger.getEffectiveLevel())  # should be <= WARNING (30)
logger.setLevel(logging.WARNING)          # force it open regardless of parent config
logger.addHandler(debug_capture)          # attach directly, don't rely on propagation
logger.propagate = True

effective level: 10


In [5]:
results = []

for mode in ['high_capacity', 'low_capacity']:
    cmd = sim.UserCommands(use_instrument="METIS", set_modes=[f'wcu_lss_n'],
                           properties={'!OBS.detector_readout_mode': mode})
    cmd['!OBS.auto_exposure.fill_frac'] = 0.75

    opt_train = sim.OpticalTrain(cmd)

    nd_filters = list(opt_train['nd_filter_wheel'].filters.keys())
    nd_filters.remove('closed')

    slits = list(opt_train['slit_wheel'].slits.keys())

    for nd_filt in nd_filters:
        opt_train['nd_filter_wheel'].change_filter(nd_filt)
        for slit in slits:
            opt_train['slit_wheel'].change_slit(slit)

            opt_train.observe()

            debug_capture.raw.clear()

            outhdul = opt_train.readout(exptime=3600)[0]

            attempted_dit = None
            mindit = None
            for msg in debug_capture.raw:
                match = re.search(r"DIT\s*=\s*([\d.]+)\s*s\s*<\s*MINDIT\s*=\s*([\d.]+)", msg)
                if match:
                    attempted_dit = float(match.group(1))
                    mindit = float(match.group(2))
                    break

            det_no = 2 
            gain = outhdul[1].header[f"ESO DET{det_no} CHIP GAIN"] * u.electron / u.adu
            full_well = outhdul[1].header[f"ESO DET{det_no} CHIP FULLWELL"] * u.electron
            outimg = outhdul[1].data * u.adu * gain
            fill_frac = outimg.max() / full_well << u.percent
            dit = outhdul[0].header['HIERARCH ESO DET DIT']


            results.append({
                'nd_filter': nd_filt,
                'detmode': mode,
                'slit': slit,
                'dit': dit,
                'fill_frac': fill_frac,
                'mindit_triggered': attempted_dit is not None,
                'attempted_dit': attempted_dit,
                'mindit': mindit,
            })

results_df = pd.DataFrame(results)

results_df.to_csv(f"N_LSS_results.csv")

py.warnings - WARNING: /home/jessicap/Desktop/Packages/ScopeSim/scopesim/effects/spectral_trace_list.py:169: DeprecationWarning: The fov_grid method is deprecated and will be removed in a future release.
  fov = thetrace.fov_grid()

py.warnings - WARNING: /home/jessicap/Desktop/Packages/ScopeSim/scopesim/effects/spectral_trace_list.py:224: DeprecationWarning: The fov_grid method is deprecated and will be removed in a future release.
  vol = spt.fov_grid()

py.warnings - WARNING: /home/jessicap/Desktop/Packages/ScopeSim/scopesim/effects/spectral_trace_list.py:224: DeprecationWarning: The fov_grid method is deprecated and will be removed in a future release.
  vol = spt.fov_grid()

astar.scopesim.optics.optical_train - Observing empty field


 FOVs:   0%|          | 0/29 [00:00<?, ?it/s]

 FOV effects:   0%|          | 0/2 [00:00<?, ?it/s]

astar.scopesim.effects.psfs - Interpolating PSF onto (252, 7, 256) cube
astar.scopesim.effects.psfs - Interpolation order 1


 PSF slices:   0%|          | 0/252 [00:00<?, ?it/s]

...

astar.scopesim.optics.image_plane - No BUNIT found in added HDU.


 FOV effects:   0%|          | 0/2 [00:00<?, ?it/s]

astar.scopesim.effects.psfs - Interpolating PSF onto (138, 35, 256) cube
astar.scopesim.effects.psfs - Interpolation order 1


 PSF slices:   0%|          | 0/138 [00:00<?, ?it/s]

astar.scopesim.optics.image_plane - No BUNIT found in added HDU.
astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
